In [1]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [2]:
from google.colab import files
uploaded = files.upload()

Saving data.tar.gz to data.tar.gz


In [3]:
!tar -xzvf data.tar.gz

._data
data/
data/claims_dev.jsonl
data/cross_validation/
data/claims_train.jsonl
data/claims_test.jsonl
data/corpus.jsonl
data/cross_validation/._fold_4
data/cross_validation/fold_4/
data/cross_validation/fold_3/
data/cross_validation/fold_2/
data/cross_validation/fold_5/
data/cross_validation/._fold_1
data/cross_validation/fold_1/
data/cross_validation/fold_1/claims_dev_1.jsonl
data/cross_validation/fold_1/claims_train_1.jsonl
data/cross_validation/fold_5/claims_train_5.jsonl
data/cross_validation/fold_5/claims_dev_5.jsonl
data/cross_validation/fold_2/claims_dev_2.jsonl
data/cross_validation/fold_2/claims_train_2.jsonl
data/cross_validation/fold_3/claims_train_3.jsonl
data/cross_validation/fold_3/claims_dev_3.jsonl
data/cross_validation/fold_4/claims_dev_4.jsonl
data/cross_validation/fold_4/claims_train_4.jsonl


In [4]:
!find . -name "*.jsonl"

./data/claims_train.jsonl
./data/cross_validation/fold_2/claims_dev_2.jsonl
./data/cross_validation/fold_2/claims_train_2.jsonl
./data/cross_validation/fold_3/claims_dev_3.jsonl
./data/cross_validation/fold_3/claims_train_3.jsonl
./data/cross_validation/fold_5/claims_dev_5.jsonl
./data/cross_validation/fold_5/claims_train_5.jsonl
./data/cross_validation/fold_4/claims_train_4.jsonl
./data/cross_validation/fold_4/claims_dev_4.jsonl
./data/cross_validation/fold_1/claims_train_1.jsonl
./data/cross_validation/fold_1/claims_dev_1.jsonl
./data/corpus.jsonl
./data/claims_test.jsonl
./data/claims_dev.jsonl


In [5]:
import json

def load_corpus(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_claims(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    return [json.loads(l) for l in content.splitlines() if l.strip()]

corpus = load_corpus("data/corpus.jsonl")
claims_train = load_claims("data/claims_train.jsonl")
claims_dev   = load_claims("data/claims_dev.jsonl")
claims_test  = load_claims("data/claims_test.jsonl")
corpus_by_id = {p["doc_id"]: p for p in corpus}

print(f"Corpus papers: {len(corpus)}")
print(f"Train claims: {len(claims_train)}")

Corpus papers: 5183
Train claims: 809


In [6]:
def build_nli_examples(claims, corpus_by_id):
    examples = []
    for claim in claims:
        claim_text = claim["claim"]
        evidence = claim.get("evidence", {})

        if not evidence:
            examples.append({
                "claim": claim_text,
                "evidence_text": "",
                "label": "NOT_ENOUGH_INFO"
            })
            continue

        for doc_id_str, ev_list in evidence.items():
            doc_id = int(doc_id_str)
            paper = corpus_by_id.get(doc_id)
            if paper is None:
                continue
            for ev in ev_list:
                sent_ids = ev["sentences"]
                evidence_text = " ".join([paper["abstract"][i] for i in sent_ids])
                label = ev["label"]
                examples.append({
                    "claim": claim_text,
                    "evidence_text": evidence_text,
                    "label": label
                })
    return examples

train_examples = build_nli_examples(claims_train, corpus_by_id)
dev_examples = build_nli_examples(claims_dev, corpus_by_id)

print(f"Train examples: {len(train_examples)}")
print(f"Dev examples: {len(dev_examples)}")
print(train_examples[1])

Train examples: 1261
Dev examples: 450
{'claim': '1 in 5 million in UK have abnormal PrP positivity.', 'evidence_text': 'RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).', 'label': 'CONTRADICT'}


In [7]:
label2id = {"SUPPORT": 0, "CONTRADICT": 1, "NOT_ENOUGH_INFO": 2}
id2label = {v: k for k, v in label2id.items()}
for ex in train_examples:
    ex["label_id"] = label2id[ex["label"]]
for ex in dev_examples:
    ex["label_id"] = label2id[ex["label"]]

In [8]:
def tokenize_examples(examples, tokenizer, max_length=256):
    claims = [ex["claim"] for ex in examples]
    evidences = [ex["evidence_text"] for ex in examples]
    labels = [ex["label_id"] for ex in examples]

    encodings = tokenizer(
        claims,
        evidences,
        truncation=True,
        padding=True,
        max_length=max_length
    )
    return encodings, labels

In [9]:
from transformers import AutoTokenizer
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings, train_labels = tokenize_examples(train_examples, tokenizer)
dev_encodings, dev_labels = tokenize_examples(dev_examples, tokenizer)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

In [10]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels
})
dev_dataset = Dataset.from_dict({
    "input_ids": dev_encodings["input_ids"],
    "attention_mask": dev_encodings["attention_mask"],
    "labels": dev_labels
})
print("Train dataset size:", len(train_dataset))
print("Dev dataset size:", len(dev_dataset))

Train dataset size: 1261
Dev dataset size: 450


In [11]:
!pip install transformers -q

In [12]:
from transformers import AutoModelForSequenceClassification
model_name = "allenai/scibert_scivocab_uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
for name, param in model.named_parameters():
    print(name)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  442MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [13]:
for param in model.parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name or "pooler" in name or "classifier" in name:
        param.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

Trainable parameters: 14,768,643 / 109,920,771


In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_partial",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
)

In [15]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    f1_macro = f1_score(p.label_ids, preds, average="macro")
    acc = accuracy_score(p.label_ids, preds)
    return {"accuracy": acc, "f1_macro": f1_macro}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.539938,0.628244,0.637778,0.594341
2,0.582632,0.599363,0.600000,0.617792
3,0.498904,0.574375,0.637778,0.655391
4,0.341480,0.561456,0.715556,0.686830
5,0.360210,0.562689,0.702222,0.684197
6,0.316246,0.593651,0.688889,0.686039
7,0.346133,0.596979,0.695556,0.708057
8,0.267419,0.601964,0.684444,0.694330


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=632, training_loss=0.42968948458946205, metrics={'train_runtime': 312.0896, 'train_samples_per_second': 32.324, 'train_steps_per_second': 2.025, 'total_flos': 1285670826550656.0, 'train_loss': 0.42968948458946205, 'epoch': 8.0})

In [17]:
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [18]:
!zip -r final_model.zip final_model
from google.colab import files
files.download("final_model.zip")

  adding: final_model/ (stored 0%)
  adding: final_model/model.safetensors (deflated 7%)
  adding: final_model/tokenizer.json (deflated 71%)
  adding: final_model/config.json (deflated 54%)
  adding: final_model/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
test_examples = build_nli_examples(claims_test, corpus_by_id)
for ex in test_examples:
    ex["label_id"] = label2id[ex["label"]]
print(f"Test examples: {len(test_examples)}")

Test examples: 300


In [20]:
test_encodings, test_labels = tokenize_examples(test_examples, tokenizer)
from datasets import Dataset
test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels
})

In [21]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.267419,0.043087,8,0.980000,0.329966


{'eval_loss': 0.043086666613817215, 'eval_accuracy': 0.98, 'eval_f1_macro': 0.32996632996632996}


In [22]:
from collections import Counter
print(Counter([ex["label"] for ex in test_examples]))

Counter({'NOT_ENOUGH_INFO': 300})


In [23]:
dev_results = trainer.evaluate(dev_dataset)
print(dev_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.267419,0.596979,8,0.695556,0.708057


{'eval_loss': 0.5969785451889038, 'eval_accuracy': 0.6955555555555556, 'eval_f1_macro': 0.7080565835546334}


In [24]:
import torch

def predict_verdict(claim_text, evidence_text, model=model, tokenizer=tokenizer, id2label=id2label):
    inputs = tokenizer(claim_text, evidence_text, truncation=True, padding=True, max_length=256, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

    return {
        "verdict": id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "all_probs": {id2label[i]: float(probs[i]) for i in range(len(probs))}
    }

In [25]:
example = dev_examples[1]
result = predict_verdict(example["claim"], example["evidence_text"])
print("Claim:", example["claim"])
print("Evidence:", example["evidence_text"])
print("True label:", example["label"])
print("Predicted:", result)

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
Evidence: We propose as an alternative explanation that variants much less common than the associated one may create "synthetic associations" by occurring, stochastically, more often in association with one of the alleles at the common site versus the other allele. We show that they are not only possible, but inevitable, and that under simple but reasonable genetic models, they are likely to account for or contribute to many of the recently identified signals reported in genome-wide association studies.
True label: SUPPORT
Predicted: {'verdict': 'SUPPORT', 'confidence': 0.7752552628517151, 'all_probs': {'SUPPORT': 0.7752552628517151, 'CONTRADICT': 0.17808504402637482, 'NOT_ENOUGH_INFO': 0.046659670770168304}}


In [26]:
from google.colab import files
uploaded = files.upload()

Saving rationale_selector.joblib to rationale_selector.joblib


In [27]:
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

rationale_data = joblib.load("rationale_selector.joblib")
clf_evidence = rationale_data["model"]
vectorizer = rationale_data["vectorizer"]

def word_overlap(a, b):
    wa, wb = set(a.lower().split()), set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def featurize(rows):
    claim_vecs = vectorizer.transform([r["claim"] for r in rows])
    sent_vecs  = vectorizer.transform([r["sentence"] for r in rows])
    sims = np.array([cosine_similarity(claim_vecs[i], sent_vecs[i])[0][0] for i in range(len(rows))])
    overlaps = np.array([word_overlap(r["claim"], r["sentence"]) for r in rows])
    positions = np.array([r["sentence_idx"] for r in rows])
    lengths = np.array([len(r["sentence"].split()) for r in rows])
    X = np.column_stack([sims, overlaps, positions, lengths])
    return X

def select_evidence(claim_text, retrieved_paper, clf=clf_evidence, threshold=0.5):
    abstract = retrieved_paper["abstract"]
    rows = [{"claim": claim_text, "sentence": s, "sentence_idx": i} for i, s in enumerate(abstract)]
    X = featurize(rows)
    probs = clf.predict_proba(X)[:, 1]
    selected = [
        {"sentence_idx": i, "sentence": abstract[i], "probability": float(probs[i])}
        for i in range(len(abstract)) if probs[i] >= threshold
    ]
    selected.sort(key=lambda x: x["probability"], reverse=True)
    return selected

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

In [28]:
uploaded = files.upload()

Saving retrieved_docs.json to retrieved_docs.json


In [29]:
import json
with open("retrieved_docs.json") as f:
    retrieved = json.load(f)

In [30]:
def verify_claim(claim_text, claim_id, split="dev", top_k_docs=1):
    doc_ids = retrieved[split].get(str(claim_id), [])
    if not doc_ids:
        return {"verdict": "NOT_ENOUGH_INFO", "reason": ""}

    all_results = []

    for doc_id in doc_ids[:top_k_docs]:
        paper = corpus_by_id.get(doc_id)
        if paper is None:
            continue
        evidence_sentences = select_evidence(claim_text, paper, threshold=0.5)
        if not evidence_sentences:
            continue

        evidence_text = " ".join([e["sentence"] for e in evidence_sentences[:3]])
        result = predict_verdict(claim_text, evidence_text)
        result["doc_id"] = doc_id
        result["evidence_used"] = evidence_text
        all_results.append(result)

    if not all_results:
        return {"verdict": "NOT_ENOUGH_INFO", "reason": ""}

    # aggregation instead of max-confidence
    labels = all_results[0]["all_probs"].keys()
    avg_probs = {
        label: sum(r["all_probs"][label] for r in all_results) / len(all_results)
        for label in labels
    }
    verdict = max(avg_probs, key=avg_probs.get)

    return {
        "verdict": verdict,
        "confidence": avg_probs[verdict],
        "all_probs": avg_probs,
        "doc_ids": [r["doc_id"] for r in all_results],
        "evidence_used": " || ".join(r["evidence_used"] for r in all_results),
    }

In [31]:
example_claim = next(c for c in claims_dev if c.get("evidence"))
result = verify_claim(example_claim["claim"], example_claim["id"], split="dev")
print("Claim:", example_claim["claim"])
print("Result:", result)

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
Result: {'verdict': 'SUPPORT', 'confidence': 0.44287529587745667, 'all_probs': {'SUPPORT': 0.44287529587745667, 'CONTRADICT': 0.17745773494243622, 'NOT_ENOUGH_INFO': 0.3796669840812683}, 'doc_ids': [2739854], 'evidence_used': 'The fact that they tend not to identify more than a fraction of the specific causal loci has led to divergence of opinion over whether most of the variance is hidden as numerous rare variants of large effect or as common variants of very small effect.'}


In [32]:
!pip install -q gradio

In [ ]:
import gradio as gr

# ---- helper data available to the UI (built from what you already loaded above) ----
_claims_by_split = {"train": claims_train, "dev": claims_dev, "test": claims_test}

def _claim_ids_for_split(split):
    claims = _claims_by_split.get(split, [])
    return [str(c["id"]) for c in claims]

def _find_claim_text(split, claim_id):
    claims = _claims_by_split.get(split, [])
    for c in claims:
        if str(c["id"]) == str(claim_id):
            return c["claim"]
    return None

# ---- Tab 1: Quick Check (claim + evidence typed directly) ----
def ui_quick_check(claim_text, evidence_text):
    if not claim_text or not claim_text.strip():
        raise gr.Error("Please enter a claim.")
    try:
        result = predict_verdict(claim_text.strip(), (evidence_text or "").strip())
    except NameError:
        raise gr.Error("Model/tokenizer not found — run the training/model-loading cells above first.")
    except Exception as e:
        raise gr.Error(f"Prediction failed: {e}")
    return result["all_probs"], f"{result['verdict']}  ({result['confidence']*100:.1f}% confidence)"

# ---- Tab 2: Full pipeline (claim_id -> retrieval -> evidence selection -> verdict) ----
def ui_refresh_ids(split):
    ids = _claim_ids_for_split(split)
    return gr.update(choices=ids, value=(ids[0] if ids else None))

def ui_run_pipeline(split, claim_id):
    if not claim_id:
        raise gr.Error("Pick a claim id first.")
    claim_text = _find_claim_text(split, claim_id)
    if claim_text is None:
        raise gr.Error(f"Claim id {claim_id} not found in split '{split}'.")
    try:
        result = verify_claim(claim_text, claim_id, split=split)
    except NameError:
        raise gr.Error("Pipeline not ready — make sure the retrieval/rationale cells above ran successfully.")
    except KeyError:
        raise gr.Error(f"No retrieved documents for split '{split}' — check retrieved_docs.json.")
    except Exception as e:
        raise gr.Error(f"Pipeline failed: {e}")

    probs = result.get("all_probs", {})
    verdict = result.get("verdict", "NOT_ENOUGH_INFO")
    confidence = result.get("confidence")
    conf_str = f"{confidence*100:.1f}%" if confidence is not None else "n/a"
    evidence_used = result.get("evidence_used", "") or "(no supporting evidence found)"
    summary = f"**Verdict:** {verdict}  \n**Confidence:** {conf_str}"
    return claim_text, evidence_used, probs, summary

with gr.Blocks(title="Scientific Claim Verification") as demo:
    gr.Markdown("# 🔬 Scientific Claim Verification")
    with gr.Tabs():
        with gr.Tab("Quick Check"):
            gr.Markdown("Type a claim and a piece of evidence text yourself.")
            qc_claim = gr.Textbox(label="Claim", lines=2, placeholder="e.g. Vitamin D supplementation reduces the risk of respiratory infection.")
            qc_evidence = gr.Textbox(label="Evidence text", lines=4, placeholder="Paste the sentence(s) to check the claim against...")
            qc_button = gr.Button("Check", variant="primary")
            qc_label = gr.Label(label="Probabilities")
            qc_verdict = gr.Textbox(label="Verdict", interactive=False)
            qc_button.click(ui_quick_check, inputs=[qc_claim, qc_evidence], outputs=[qc_label, qc_verdict])

        with gr.Tab("Full Pipeline"):
            gr.Markdown("Pick an existing claim id — the app retrieves its document(s), selects evidence sentences, and returns a verdict.")
            with gr.Row():
                fp_split = gr.Dropdown(choices=list(_claims_by_split.keys()), value="dev", label="Split")
                fp_claim_id = gr.Dropdown(choices=_claim_ids_for_split("dev"), label="Claim id")
            fp_split.change(ui_refresh_ids, inputs=fp_split, outputs=fp_claim_id)
            fp_button = gr.Button("Run pipeline", variant="primary")
            fp_claim_text = gr.Textbox(label="Claim text", interactive=False)
            fp_evidence = gr.Textbox(label="Evidence used", lines=4, interactive=False)
            fp_label = gr.Label(label="Probabilities")
            fp_summary = gr.Markdown()
            fp_button.click(
                ui_run_pipeline,
                inputs=[fp_split, fp_claim_id],
                outputs=[fp_claim_text, fp_evidence, fp_label, fp_summary],
            )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://51aab873847ca73101.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
